## 面试问题

循环级 trace：每步记录什么才能回放与因果定位？

## 回答主线

每步 trace 记 step_id、读取的观察/来源、动作、前后状态签名、消耗、provenance。有了它才能回放、因果定位、成本归因。本 Notebook 用一个 3 步循环，第 1 步读了不可信来源的错误观察(999)污染结果，对比无 trace(只知最终错)与有 trace(沿来源定位到第一处出错步)。

## 真实案例

3 步循环逐步累加观察值，第 1 步观察来自 flaky 来源、值 999（本应约 20）。期望正确总和 35，被污染后为 1014。数据为教学循环，不代表真实系统。

In [1]:
step_observations = [  # 每步读取的观察其中一步错误。
    {"step": 0, "value": 10, "source": "trusted"},  # 第 0 步可信观察。
    {"step": 1, "value": 999, "source": "flaky"},  # 第 1 步错误观察本应约 20。
    {"step": 2, "value": 5, "source": "trusted"},  # 第 2 步可信观察。
]  # 结束观察定义。

EXPECTED_TOTAL = 10 + 20 + 5  # 正确观察下的期望累加值。
print("每步观察:", [(o["step"], o["value"], o["source"]) for o in step_observations])  # 展示每步观察。
print("期望总和(观察正确时):", EXPECTED_TOTAL)  # 展示期望结果。

每步观察: [(0, 10, 'trusted'), (1, 999, 'flaky'), (2, 5, 'trusted')]
期望总和(观察正确时): 35


## 基线（Baseline）

反面基线：无 trace，只累加并返回最终结果。结果 1014 明显错误，但无法知道是哪一步、因为读了什么观察而出错。

In [2]:
def run_no_trace(observations):  # 无 trace 的循环只累加不记录每步。
    total = 0  # 累加器。
    for o in observations:  # 逐步读取观察。
        total += o["value"]  # 累加观察值。
    return {"total": total}  # 只返回最终结果。

no_trace = run_no_trace(step_observations)  # 无 trace 运行。
print("无 trace 最终结果:", no_trace)  # 展示只知道最终总和错误。
print("无 trace 能定位哪一步出错吗: 不能")  # 展示无法定位。

无 trace 最终结果: {'total': 1014}
无 trace 能定位哪一步出错吗: 不能


## 失败案例与修正

无 trace 无法定位。修正是结构化 trace：每步记 step、读取来源与值、前后状态签名。之后可沿来源定位第一处读到不可信观察的步。

In [3]:
def run_with_trace(observations):  # 带 trace 的循环每步记录结构化条目。
    total = 0  # 累加器。
    trace = []  # 逐步 trace。
    for o in observations:  # 逐步读取观察。
        before = total  # 记录步前状态签名。
        total += o["value"]  # 累加观察值。
        entry = {"step": o["step"], "read_source": o["source"], "read_value": o["value"], "before": before, "after": total}  # 记录本步结构化 trace。
        trace.append(entry)  # 追加 trace 条目。
    return {"total": total, "trace": trace}  # 返回结果与 trace。

def locate_first_error(trace):  # 从 trace 定位第一处读到不可信来源的步。
    for e in trace:  # 逐步检查 trace。
        if e["read_source"] == "flaky":  # 命中不可信来源。
            return {"first_bad_step": e["step"], "read_value": e["read_value"]}  # 返回第一处出错步。
    return {"first_bad_step": None}  # 未发现错误步。

In [4]:
traced = run_with_trace(step_observations)  # 带 trace 运行。
located = locate_first_error(traced["trace"])  # 用 trace 定位第一处错误步。
print("带 trace 最终结果:", traced["total"])  # 展示最终总和。
for e in traced["trace"]:  # 逐条打印 trace。
    print("  trace:", e)  # 展示每步来源、值与前后状态。
print("定位到第一处错误步:", located)  # 展示定位到第 1 步及其错误观察。

带 trace 最终结果: 1014
  trace: {'step': 0, 'read_source': 'trusted', 'read_value': 10, 'before': 0, 'after': 10}
  trace: {'step': 1, 'read_source': 'flaky', 'read_value': 999, 'before': 10, 'after': 1009}
  trace: {'step': 2, 'read_source': 'trusted', 'read_value': 5, 'before': 1009, 'after': 1014}
定位到第一处错误步: {'first_bad_step': 1, 'read_value': 999}


In [5]:
print("无 trace 只知最终错:", no_trace["total"], "!=", EXPECTED_TOTAL)  # 无 trace 只知结果错。
print("有 trace 定位到步:", located["first_bad_step"], "读到错误值", located["read_value"])  # 有 trace 定位到步。
print("该错误步来源:", traced["trace"][located["first_bad_step"]]["read_source"])  # 展示错误步来源。

无 trace 只知最终错: 1014 != 35
有 trace 定位到步: 1 读到错误值 999
该错误步来源: flaky


## 结果解读

无 trace 只知总和 1014 错误、无从下手；有 trace 逐步记录来源与前后状态，`locate_first_error` 沿 `flaky` 来源定位到第 1 步、读值 999。要点：记结构化元数据与 provenance（不记原文/秘密），前后签名定位真正改状态的步。

In [6]:
assert no_trace["total"] == 1014  # 无 trace 得到被污染的总和。
assert no_trace["total"] != EXPECTED_TOTAL  # 结果确实错误。
assert located["first_bad_step"] == 1  # trace 定位到第 1 步出错。
assert located["read_value"] == 999  # 定位到该步读了错误值 999。
assert traced["trace"][1]["read_source"] == "flaky"  # 第 1 步来源为不可信。
print("全部不变量通过")  # 输出测试通过信号。

全部不变量通过
